# TerraShift: India-Exclusive Earth System & Environmental Trend Analysis

**Geographical Scope:** India (`6.0°N–37.5°N, 68.0°E–97.5°E`)  
**Datasets:** NASA POWER (1981–2025 monthly MERRA-2 & model hydrology across 12 Indian benchmark sites) and NASA Terra/MOPITT `MOP03JM` Carbon Monoxide NetCDF subset over India.  
**Methods:** IMD 4-season stratification (`JF`, `MAM`, `JJAS`, `OND`), Mann-Kendall non-parametric test, Theil-Sen slope, OLS regression, lag-1 autocorrelation, Holm-Bonferroni multiple-hypothesis correction, and 1991–2020 baseline anomaly detection.

In [1]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'backend'))
from india_analytics import analyze_india_seasons, INDIA_PHYSIOGRAPHIC_REGIONS
from nasa_service import get_parameter_metadata
from stats_engine import analyze_series

OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style='darkgrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 140
print('Environment initialized. Regions defined:', len(INDIA_PHYSIOGRAPHIC_REGIONS))

Environment initialized. Regions defined: 10


In [2]:
with open(REPO_ROOT / 'backend' / 'cases_cache.json', 'r', encoding='utf-8') as f:
    cache = json.load(f)

records = []
series_store = {}
for c in cache['cases']:
    cid = c['case_id']
    param = c['parameter']
    unit = c.get('unit') or get_parameter_metadata(param).get('unit', '')
    for role, pt_key, s_key in [('primary', 'primary_point', 'primary_series'), ('compare', 'comparison_point', 'comparison_series')]:
        pt = c[pt_key]
        sr = c[s_key]
        ts = sr['timestamps']
        vals = sr['values']
        seas = analyze_india_seasons(ts, vals, baseline_start=1991, baseline_end=2020)
        overall = analyze_series(ts, vals)
        p_stats = overall['primary_stats']
        key = f"{cid}:{role}"
        series_store[key] = {'meta': {'case_id': cid, 'location': pt['name'], 'lat': pt['lat'], 'lon': pt['lon'], 'parameter': param, 'unit': unit}, 'ts': ts, 'vals': vals, 'seas': seas, 'overall': p_stats}
        row = {
            'case_id': cid,
            'location': pt['name'],
            'lat': pt['lat'],
            'lon': pt['lon'],
            'parameter': param,
            'unit': unit,
            'n_valid': seas['eda']['n_valid'],
            'mean': seas['eda']['mean'],
            'std': seas['eda']['std'],
            'overall_sen_per_dec': p_stats['sens_slope_per_decade'],
            'overall_p_val': p_stats['p_value'],
            'overall_sig': p_stats['is_significant'],
            'top_pos_year': seas['top_positive_years'][0]['year'] if seas['top_positive_years'] else None,
            'top_pos_anom': seas['top_positive_years'][0]['anomaly'] if seas['top_positive_years'] else None,
        }
        for sc in ['JF', 'MAM', 'JJAS', 'OND']:
            sd = seas['seasons'][sc]
            row[f'{sc}_sen_dec'] = sd['sen_slope_per_decade']
            row[f'{sc}_holm_p'] = sd['holm_adjusted_p']
            row[f'{sc}_sig'] = sd['significant_holm']
        records.append(row)

df_summary = pd.DataFrame(records)
df_summary[['location', 'parameter', 'n_valid', 'mean', 'overall_sen_per_dec', 'overall_p_val', 'JF_sen_dec', 'MAM_sen_dec', 'JJAS_sen_dec', 'OND_sen_dec']]

In [3]:
# Figure 1: Annual Temperature Anomalies across Indian Thermal Benchmarks
temp_keys = [k for k, v in series_store.items() if v['meta']['parameter'] == 'T2M']
fig, ax = plt.subplots(figsize=(11, 5))
for k in temp_keys:
    anom = series_store[k]['seas']['annual_anomalies']
    yrs = [a['year'] for a in anom]
    vals = [a['anomaly'] for a in anom]
    ax.plot(yrs, vals, marker='o', markersize=3, linewidth=1.6, label=series_store[k]['meta']['location'])
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_title('India 2m Air Temperature Annual Anomalies (1981–2025 vs 1991–2020 Baseline)')
ax.set_xlabel('Year')
ax.set_ylabel('Temperature Anomaly (°C)')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / '01_india_temperature_anomalies.png')
plt.close(fig)
print('Saved 01_india_temperature_anomalies.png')

Saved 01_india_temperature_anomalies.png


In [4]:
# Figure 2: Seasonal Sen's Slope Heatmap (Normalized Z-score Slope per Decade across IMD Seasons)
heat_rows = []
for k, v in series_store.items():
    std = v['seas']['eda']['std'] or 1.0
    heat_rows.append({
        'Location': f"{v['meta']['location']} ({v['meta']['parameter']})",
        'JF (Winter)': (v['seas']['seasons']['JF']['sen_slope_per_decade'] or 0.0) / std,
        'MAM (Pre-Monsoon)': (v['seas']['seasons']['MAM']['sen_slope_per_decade'] or 0.0) / std,
        'JJAS (Monsoon)': (v['seas']['seasons']['JJAS']['sen_slope_per_decade'] or 0.0) / std,
        'OND (Post-Monsoon)': (v['seas']['seasons']['OND']['sen_slope_per_decade'] or 0.0) / std,
    })
df_heat = pd.DataFrame(heat_rows).set_index('Location')
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title("India IMD Seasonal Sen's Slope (Standardized SD/Decade, 1981–2025)")
plt.tight_layout()
plt.savefig(OUT_DIR / '02_india_seasonal_trends_heatmap.png')
plt.close(fig)
print('Saved 02_india_seasonal_trends_heatmap.png')

Saved 02_india_seasonal_trends_heatmap.png


In [5]:
# Figure 3: Monsoon Precipitation & Soil Moisture Contrasts
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for k in ['case_1_monsoon:primary', 'case_1_monsoon:compare']:
    anom = series_store[k]['seas']['annual_anomalies']
    axes[0].plot([a['year'] for a in anom], [a['annual_mean'] for a in anom], label=series_store[k]['meta']['location'])
axes[0].set_title('Annual Mean Precipitation (Western Ghats vs Marathwada)')
axes[0].set_ylabel('mm/day')
axes[0].legend(fontsize=8)

for k in ['case_3_igp_thar:primary', 'case_3_igp_thar:compare']:
    anom = series_store[k]['seas']['annual_anomalies']
    axes[1].plot([a['year'] for a in anom], [a['annual_mean'] for a in anom], label=series_store[k]['meta']['location'])
axes[1].set_title('Root-Zone Soil Wetness (Central IGP vs Thar Desert)')
axes[1].set_ylabel('Fraction (0–1)')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / '03_india_monsoon_soil_moisture.png')
plt.close(fig)
print('Saved 03_india_monsoon_soil_moisture.png')

Saved 03_india_monsoon_soil_moisture.png


In [6]:
# Figure 4: NASA Terra/MOPITT Carbon Monoxide (MOP03JM) Subset over India
nc_path = REPO_ROOT / 'data' / 'processed' / 'india' / 'MOP03JM_CO_India_202401_202403.nc'
ds = xr.open_dataset(nc_path)
print('NetCDF Variables:', list(ds.data_vars))
var_name = list(ds.data_vars)[0]
da = ds[var_name]
mean_2d = da.mean(dim='time', skipna=True) if 'time' in da.dims else da
fig, ax = plt.subplots(figsize=(8, 6))
mean_2d.plot(ax=ax, cmap='inferno')
ax.set_title(f'NASA MOPITT {var_name} Mean over India (Jan–Mar 2024)')
plt.tight_layout()
plt.savefig(OUT_DIR / '04_india_mopitt_co_spatial.png')
plt.close(fig)
co_stats = {'var_name': var_name, 'min': float(mean_2d.min()), 'max': float(mean_2d.max()), 'mean': float(mean_2d.mean())}
ds.close()

# Save exact computed summary for INDIA_ENVIRONMENTAL_REPORT.md
summary_payload = {'locations': records, 'mopitt_co': co_stats}
with open(OUT_DIR / 'summary_stats.json', 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=2)
print('Saved 04_india_mopitt_co_spatial.png and summary_stats.json')

NetCDF Variables: ['CO_Total_Column']
Saved 04_india_mopitt_co_spatial.png and summary_stats.json
